# KG Pipeline - Notebook Only (Sections 1-14)

This notebook keeps the first five KG stages entirely in memory and now delegates the heavier analysis stages to reusable Python modules under `src/telegram_scraper/analysis/`.
It fetches Telegram channel messages, translates them, embeds them, and exposes notebook-friendly DataFrames and figure objects without writing to Postgres, Redis, or Pinecone.

Use this when you want to stop before node resolution and projection, then inspect the channel's emotional arc, thematic landscape, messaging cadence, vocabulary shifts over time, co-mentioned political actors, rhetorical framing, reply-threading behavior, phrase-level collocation networks, and media-vs-text editorial differences.


## Prerequisites

- Telegram credentials in `.env`: `TG_API_ID`, `TG_API_HASH`, `TG_PHONE`, `SESSION_PATH`
- OpenAI credentials in `.env`: `OPENAI_API_KEY`, `KG_TRANSLATION_MODEL`, `EMBEDDING_MODEL`
- Project dependencies installed so `telethon`, `openai`, `pandas`, and notebook support are available
- Optional Section 6 analysis dependencies installed: `transformers`, `torch`, `matplotlib`, `seaborn`, `numpy`
- Optional Section 7 topic-modeling dependencies installed: `umap-learn`, `hdbscan`, `plotly`, `scikit-learn`
- Optional Section 8 lexical-shift dependencies installed: `nltk`, `scikit-learn`, `matplotlib`, `wordcloud`
- Optional Section 9 named-entity dependencies installed: `spacy`, `networkx`, `plotly`, `python-louvain`, `matplotlib`, plus a spaCy English model such as `en_core_web_sm`
- Optional Section 10 cadence dependencies reuse `matplotlib`, `seaborn`, and `numpy` from the Section 6 stack
- Optional Section 11 rhetoric dependencies reuse `transformers`, `torch`, `plotly`, `matplotlib`, and `seaborn` from Sections 6-10
- Optional Section 12 reply-threading dependencies reuse `networkx` from Section 9 plus `scipy` and `matplotlib` from earlier analysis sections
- Optional Section 13 phrase-network dependencies installed: `nltk`, `networkx`, `plotly`, `matplotlib`, `numpy`
- Optional Section 14 comparison dependencies reuse `scikit-learn`, `numpy`, `matplotlib`, and `seaborn` from earlier sections and also require `scipy`

## Write targets

| # | Stage | Writes to |
|---|---|---|
| 1 | Setup | - |
| 2 | Telegram Client & Channel Discovery | local session file, optional media under `tmp/notebook-media/` |
| 3 | Message Ingestion | in-memory Python objects only |
| 4 | Translation | OpenAI API response only, held in memory |
| 5 | Embedding | OpenAI API response only, held in memory |
| 6 | Sentiment & Emotion Over Time | Hugging Face model cache plus in-memory analysis DataFrames and plots |
| 7 | Topic Modeling Landscape | in-memory topic-modeling DataFrames plus interactive Plotly figures |
| 8 | Word Frequency & TF-IDF Shifts | in-memory lexical-analysis DataFrames plus matplotlib figures and word clouds |
| 9 | Named Entity Network Graphs | in-memory entity tables, NetworkX graph object, Plotly figures, and matplotlib ego-network panels |
| 10 | Messaging Cadence & Volume Heatmaps | in-memory cadence DataFrames plus matplotlib / seaborn figures |
| 11 | Framing & Rhetoric Analysis | in-memory rhetoric DataFrames plus Plotly / matplotlib figures |
| 12 | Reply Threading & Engagement Proxy Analysis | in-memory reply-graph DataFrames, NetworkX graph object, and matplotlib figures |
| 13 | Bigram / Trigram Phrase Networks | in-memory phrase tables, NetworkX graph object, Plotly figures, and a matplotlib temporal-comparison panel |
| 14 | Media vs. Text-Only Comparison | in-memory comparison DataFrames plus matplotlib / seaborn figures |

The notebook leaves you with `raw_messages`, `translated_messages`, `messages_df`, `translation_df`, `embedding_lookup`, `embeddings_df`, `df_text`, `sentiment_emotion_df`, `sentiment_window_df`, `emotion_window_df`, `candidate_events_df`, `topic_messages_df`, `topic_keyword_df`, `topic_summary_df`, `topic_prevalence_df`, `topic_daily_share_df`, `topic_example_messages_df`, `topic_scatter_fig`, `topic_prevalence_fig`, `topic_time_fig`, `tfidf_messages_df`, `tfidf_period_docs_df`, `tfidf_score_df`, `tfidf_rank_df`, `tfidf_risers_df`, `tfidf_fallers_df`, `tfidf_movers_df`, `tfidf_top_terms_df`, `tfidf_bar_plot_df`, `tfidf_wordcloud_frequencies`, `tfidf_bump_fig`, `tfidf_terms_fig`, `tfidf_wordcloud_fig`, `entity_messages_df`, `entity_mentions_df`, `entity_summary_df`, `entity_pair_df`, `entity_network_summary_df`, `entity_network_nodes_df`, `entity_network_edges_df`, `entity_community_summary_df`, `named_entity_graph`, `entity_top_entities_fig`, `entity_network_fig`, `entity_ego_fig`, `cadence_messages_df`, `cadence_hourly_counts_df`, `cadence_daily_summary_df`, `cadence_top_spikes_df`, `cadence_calendar_heatmap_fig`, `cadence_structural_rhythm_fig`, `cadence_volume_media_fig`, `rhetoric_df_text`, `rhetoric_messages_df`, `rhetoric_window_df`, `rhetoric_window_long_df`, `rhetoric_label_counts_df`, `rhetoric_half_summary_df`, `rhetoric_half_flow_df`, `rhetoric_taxonomy_df`, `rhetoric_example_messages_df`, `rhetoric_validation_sample_df`, `rhetoric_sentiment_crosstab_df`, `rhetoric_sentiment_share_df`, `rhetoric_frame_label_lookup`, `rhetoric_frame_color_map`, `rhetoric_over_time_fig`, `rhetoric_transition_fig`, `rhetoric_examples_fig`, `rhetoric_sentiment_heatmap_fig`, `reply_messages_df`, `reply_edges_df`, `reply_top_replied_df`, `reply_thread_summary_df`, `reply_distribution_df`, `reply_hourly_reply_rate_df`, `reply_feature_tests_df`, `reply_media_contingency_df`, `reply_first_reply_timing_df`, `reply_content_review_df`, `reply_summary_df`, `reply_graph`, `reply_distribution_fig`, `reply_threads_fig`, `reply_scatter_fig`, `reply_timing_fig`, `phrase_messages_df`, `phrase_bigram_df`, `phrase_trigram_df`, `phrase_bigram_bar_df`, `phrase_network_summary_df`, `phrase_network_nodes_df`, `phrase_network_edges_df`, `phrase_temporal_bigram_df`, `phrase_temporal_change_df`, `phrase_bigram_graph`, `phrase_network_fig`, `phrase_bigram_bar_fig`, `phrase_temporal_fig`, `media_text_messages_df`, `media_text_segment_summary_df`, `media_text_stat_tests_df`, `media_text_hourly_distribution_df`, `media_text_topic_distribution_df`, `media_text_frame_distribution_df`, `media_text_tfidf_terms_df`, `media_text_entity_distribution_df`, `media_text_summary_df`, `media_text_dashboard_fig`, `media_text_violin_fig`, `media_text_hour_density_fig`, `media_text_topic_fig`, and `media_text_frame_fig`.


---
## Section 1 - Environment Setup

**Methodology.** This section standardizes the notebook runtime before any Telegram data is touched. It resolves the project root, adds `src/` to `sys.path`, applies `nest_asyncio` so notebook cells can `await` Telethon coroutines, and configures pandas display options for wide audit tables. The second cell loads `.env` into `Settings` and `NotebookSettings`, validates that Telegram / OpenAI credentials are present, creates the notebook media directory, and instantiates the reusable translation and embedding clients.

**How to read the output.** Treat the printed paths and model names as a preflight checklist. If the project root, `.env` path, session path, notebook media root, translation model, or embedding model are wrong here, fix them before moving on because every later section depends on these handles.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

try:
    import nest_asyncio
except ImportError:
    nest_asyncio = None

if nest_asyncio is not None:
    nest_asyncio.apply()

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 180)

_nb_dir = Path.cwd().resolve()
_project_root = _nb_dir if (_nb_dir / "src").exists() else _nb_dir.parent
_src_path = _project_root / "src"
if not _src_path.exists():
    raise RuntimeError(f"Could not locate src/ from notebook cwd: {_nb_dir}")
if str(_src_path) not in sys.path:
    sys.path.insert(0, str(_src_path))


def _close_matplotlib_figures(*figures) -> None:
    try:
        from matplotlib.figure import Figure as MatplotlibFigure
        import matplotlib.pyplot as plt
    except ImportError:
        return

    for figure in figures:
        if isinstance(figure, MatplotlibFigure):
            plt.close(figure)


print(f"Project root: {_project_root}")
print(f"Python: {sys.version.split()[0]}")


In [ ]:
from telegram_scraper.config import Settings
from telegram_scraper.notebook_pipeline import NotebookSettings, OpenAIEmbedder, OpenAIMessageTranslator

_t1 = time.monotonic()

_env_candidates = [_project_root / ".env", _nb_dir / ".env"]
_env_path = next((p for p in _env_candidates if p.exists()), _project_root / ".env")

settings = Settings.load(_env_path)
notebook_settings = NotebookSettings.load(_env_path)

settings.require_credentials()
notebook_settings.require_translation()
notebook_settings.require_embeddings()

_session_path = settings.session_path if settings.session_path.is_absolute() else (_project_root / settings.session_path).resolve()
notebook_media_root = (_project_root / "tmp" / "notebook-media").resolve()
notebook_media_root.mkdir(parents=True, exist_ok=True)

translator = OpenAIMessageTranslator(
    api_key=notebook_settings.openai_api_key,
    model=notebook_settings.translation_model,
    max_chars=notebook_settings.semantic_max_chars,
    batch_size=notebook_settings.semantic_batch_size,
)
embedder = OpenAIEmbedder(
    api_key=notebook_settings.openai_api_key,
    model=notebook_settings.embedding_model,
)

print(f"Loaded settings from: {_env_path}")
print(f"Telegram session path: {_session_path}")
print(f"Notebook media root: {notebook_media_root}")
print(f"Translation model: {notebook_settings.translation_model}")
print(f"Embedding model: {notebook_settings.embedding_model}")
print(f"Section 1 done in {time.monotonic() - _t1:.2f}s")

---
## Section 2 - Telegram Client & Channel Discovery

**Methodology.** This section authenticates through the saved Telethon session, requests the account's visible dialogs, normalizes them into internal chat records, and filters them down to channels only. Include / exclude filters from `.env` are applied here, so the table is the exact universe of channels available for selection in the rest of the notebook.

**How to read the output.** `channels_df` is a selection table, not an analysis result. Confirm that the target channel appears, note its row index, and use that index in the next cell. If a channel is missing, the issue is usually authentication, account visibility, or an overly restrictive include / exclude filter.


In [ ]:
from telegram_scraper.chat_discovery import discover_chats, filter_chats
from telegram_scraper.models import ChatType
from telegram_scraper.telegram_client import TelegramAccountClient

_t2 = time.monotonic()

telegram_client = TelegramAccountClient(
    api_id=settings.api_id or 0,
    api_hash=settings.api_hash,
    phone=settings.phone,
    session_path=_session_path,
    output_root=notebook_media_root,
)

await telegram_client.connect()
dialogs = await telegram_client.get_dialogs()
all_chats = discover_chats(dialogs)
channels = filter_chats(
    all_chats,
    chat_types=(ChatType.CHANNEL,),
    include_chats=settings.include_chats,
    exclude_chats=settings.exclude_chats,
)

channels_df = pd.DataFrame(
    [
        {
            "#": i,
            "chat_id": chat.chat_id,
            "title": chat.title or "",
            "username": chat.username or "",
            "slug": chat.slug or "",
        }
        for i, chat in enumerate(channels)
    ]
)

print(f"Found {len(channels)} channels in {time.monotonic() - _t2:.2f}s")
display(channels_df)

In [ ]:
CHANNEL_INDEX = 0
MESSAGE_LIMIT = 1200

if not channels:
    raise RuntimeError("No channels matched the current include/exclude filters.")
if CHANNEL_INDEX < 0 or CHANNEL_INDEX >= len(channels):
    raise IndexError(f"CHANNEL_INDEX={CHANNEL_INDEX} is out of range for {len(channels)} channels.")

selected_chat = channels[CHANNEL_INDEX]
print(f"Selected channel: [{selected_chat.chat_id}] {selected_chat.title or '(untitled)'}")
print(f"Username: @{selected_chat.username}" if selected_chat.username else "Username: ---")
print(f"Message limit: {MESSAGE_LIMIT}")

---
## Section 3 - Message Ingestion

**Methodology.** The notebook now pulls raw Telegram message envelopes for the selected channel, keeps the original metadata, and converts each message into a normalized in-memory record with consistent fields such as `timestamp`, `reply_to_message_id`, forwarding info, and media references. The records are then sorted chronologically so all later time-series and thread analyses use a stable ordering.

**How to read the output.** `messages_df` is the raw audit layer for the notebook. Check the timestamp range, reply coverage, media prevalence, and sample text quality here before running translation or modeling. If this preview looks wrong, every downstream section will inherit the problem.


In [ ]:
from telegram_scraper.notebook_pipeline import normalize_message_record

_t3 = time.monotonic()

raw_envelopes = []
async for envelope in telegram_client.iter_message_envelopes(
    selected_chat,
    limit=MESSAGE_LIMIT,
    reverse=False,
):
    raw_envelopes.append(envelope)

raw_messages = [normalize_message_record(env.record, raw_json=env.raw_json) for env in raw_envelopes]
raw_messages.sort(key=lambda message: (message.timestamp, message.message_id))

messages_df = pd.DataFrame(
    [
        {
            "channel_id": message.channel_id,
            "message_id": message.message_id,
            "timestamp": message.timestamp,
            "sender_name": message.sender_name or "",
            "reply_to_message_id": message.reply_to_message_id,
            "text": (message.text or "")[:200],
            "has_media": bool(message.media_refs),
        }
        for message in raw_messages
    ]
)

print(f"Fetched {len(raw_messages)} messages in {time.monotonic() - _t3:.2f}s")
display(messages_df.head(20))

---
## Section 4 - Translation

**Methodology.** This section batch-translates message text into English while preserving the original text, detected source language, and a `used_translation` flag that shows whether the English field actually changed the message. Media-only posts stay untouched. The goal is to create a common analytic surface so later NLP sections compare messages in one language without losing the original wording.

**How to read the outputs.** `language_counts_df` tells you how multilingual the sample is, and `translation_df` is the spot-check table for quality control. Review a few rows before continuing because Sections 6-14 interpret the translated English text as the main text field whenever it is available.


In [ ]:
_t4 = time.monotonic()

translated_messages = translator.translate_messages(raw_messages)

translation_df = pd.DataFrame(
    [
        {
            "message_id": message.message_id,
            "timestamp": message.timestamp,
            "source_language": message.source_language or "und",
            "original_text": (message.text or "")[:160],
            "english_text": (message.english_text or "")[:160],
            "used_translation": bool(
                (message.english_text or "").strip()
                and (message.english_text or "").strip() != (message.text or "").strip()
            ),
        }
        for message in translated_messages
    ]
)

language_counts_df = (
    translation_df["source_language"]
    .value_counts()
    .rename_axis("source_language")
    .reset_index(name="message_count")
)

print(f"Translated {len(translated_messages)} messages in {time.monotonic() - _t4:.2f}s")
display(language_counts_df)
display(translation_df.head(20))

---
## Section 5 - Embedding

**Methodology.** Embeddings are created from the preferred analysis text for each message—translated English when available, otherwise the original text. Each text is truncated to the configured semantic limit, embedded in batches, and stored in `embedding_lookup` keyed by `(channel_id, message_id)`. Those vectors are the reusable semantic layer for topic modeling and any later similarity-based inspection.

**How to read the output.** `embeddings_df` shows coverage, dimensionality, and a short numeric preview only; embeddings are not meant to be read directly. The main checks are whether most text-bearing messages received vectors and whether the embedding dimension matches the configured model.


In [ ]:
from telegram_scraper.notebook_pipeline import preferred_message_text, safe_message_text

_t5 = time.monotonic()

embeddable_messages = [
    message
    for message in translated_messages
    if (message.english_text or message.text or "").strip()
]

embed_texts = [
    safe_message_text(
        preferred_message_text(message) or "(media only telegram message)",
        max_chars=notebook_settings.semantic_max_chars,
    )
    for message in embeddable_messages
]

vectors = embedder.embed_texts(embed_texts) if embed_texts else []
embedding_lookup = {
    (message.channel_id, message.message_id): vector
    for message, vector in zip(embeddable_messages, vectors)
}
embedding_records = [
    {
        "channel_id": message.channel_id,
        "message_id": message.message_id,
        "timestamp": message.timestamp,
        "embedding": vector,
    }
    for message, vector in zip(embeddable_messages, vectors)
]

embeddings_df = pd.DataFrame(
    [
        {
            "channel_id": message.channel_id,
            "message_id": message.message_id,
            "timestamp": message.timestamp,
            "embedding_dim": len(vector),
            "embedding_preview": [round(value, 4) for value in vector[:8]],
        }
        for message, vector in zip(embeddable_messages, vectors)
    ]
)

print(f"Embedded {len(embeddable_messages)} messages in {time.monotonic() - _t5:.2f}s")
print(f"Embedding dimensionality: {len(vectors[0]) if vectors else 0}")
display(embeddings_df.head(20))

---
## Section 6 - Sentiment & Emotion Over Time

This section implements `docs/01_sentiment_emotion_over_time.md` directly inside the notebook.

**Methodology.** After light cleaning (URLs, emoji, and trailing delimiters), each text-bearing message is scored with two transformer models: a 3-way sentiment classifier and a 7-way emotion classifier. Sentiment is converted into a continuous score as `positive - negative`, then aggregated into 6-hour windows with a rolling mean and 95% confidence interval. Dominant emotions are summarized both as 6-hour shares for the timeline and as 1-hour labels for the heatmap and extreme-hour callout.

**Why this design is useful.** The 6-hour window smooths noisy message-level predictions into a readable narrative arc, while the 1-hour layer preserves event-level volatility. The first run downloads both Hugging Face models, so CPU execution can take a few minutes.


In [ ]:
from telegram_scraper.analysis.sentiment import (
    SentimentEmotionConfig,
    run_sentiment_emotion_analysis,
)

EVENT_ANNOTATIONS = [
    # Replace these placeholders after reviewing candidate_events_df.
    # {"timestamp": "2026-04-10 12:00:00+00:00", "label": "Iran-US peace talks begin"},
    # {"timestamp": "2026-04-11 18:00:00+00:00", "label": "Trump threatens Iranian infrastructure"},
]

sentiment_config = SentimentEmotionConfig()
print(
    f"Section 6 configured with {sentiment_config.window_freq} aggregation windows "
    f"and {sentiment_config.heatmap_freq} heatmap bins."
)


In [ ]:
sentiment_result = run_sentiment_emotion_analysis(
    translated_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    event_annotations=EVENT_ANNOTATIONS,
    config=sentiment_config,
)
globals().update(sentiment_result.to_namespace())

print(f"Section 6 completed in {sentiment_result.analysis_seconds:.2f}s")
if not EVENT_ANNOTATIONS:
    print(
        "EVENT_ANNOTATIONS is empty; using candidate placeholders. "
        "Replace them with reviewed event labels for the final chart."
    )

_close_matplotlib_figures(sentiment_over_time_fig, emotion_heatmap_fig)


In [ ]:
display(overall_summary_df)
display(sentiment_label_counts_df)
display(emotion_label_counts_df)
display(candidate_events_df)

In [ ]:
display(
    sentiment_emotion_df[
        [
            "timestamp",
            "sentiment_score",
            "dominant_sentiment",
            "dominant_emotion",
            "emotion_confidence",
            "text",
        ]
    ].head(10)
)

In [ ]:
display(event_annotations_df)

In [ ]:
display(sentiment_over_time_fig)

**How to read the timeline**

- Left axis: sentiment score from `-1` to `+1`, computed as `positive - negative`.
- Thin gray line: raw 6-hour mean sentiment. Thick black line: rolling mean across those windows. Shaded band: 95% confidence interval around the 6-hour mean.
- Stacked area on the secondary axis: the share of messages in each 6-hour window whose dominant emotion is anger, disgust, fear, joy, sadness, surprise, or neutral.
- Vertical numbered markers: reviewed event annotations. The outlined marker is different—it marks the single most extreme **1-hour** mean sentiment, so it can sit above or below the smoother 6-hour curve.

When writing findings, rely on the live tables (`most_extreme_hour`, `candidate_events_df`, `event_annotations_df`, and `sentiment_window_df`) instead of hard-coding one run's numbers into the markdown.


In [ ]:
display(emotion_heatmap_fig)
display(hourly_dominant_emotion_df.head(24))

**How to read the hourly emotion heatmap**

- Each row is a date and each column is an hour in UTC.
- Cell color is the dominant emotion for that hour; the accompanying table `hourly_dominant_emotion_df` gives the exact labels behind the colors.
- Long blocks of the same color indicate a stable tone across the day, while abrupt color changes show rapid emotional pivots.
- Empty / no-data hours are useful too: they distinguish genuine emotional quiet from simply not posting in that hour.


---
## Section 7 - Topic Modeling Landscape

This section implements `docs/02_topic_modeling_landscape.md` directly inside the notebook.

**Methodology.** Section 5 embeddings are reused as the semantic feature space. Cleaned message text is paired with each embedding, projected into two dimensions with UMAP, clustered with HDBSCAN, and then summarized with c-TF-IDF-style keywords plus representative example messages. Topic labels default to keyword-based placeholders, but `TOPIC_LABEL_OVERRIDES` lets you rename clusters after you inspect the summaries.

**Interpretation guardrail.** UMAP preserves local neighborhoods better than exact global geometry. Points that sit close together are usually semantically similar, but the absolute distance between far-apart clusters should not be over-interpreted.

Run Sections 1-5 first. Section 6 is not required for this section.


In [ ]:
from telegram_scraper.analysis.topics import TopicModelingConfig, run_topic_modeling_analysis

topic_config = TopicModelingConfig()
print(
    "Section 7 configured with "
    f"UMAP neighbors={topic_config.umap_neighbors}, "
    f"min_dist={topic_config.umap_min_dist}, "
    f"HDBSCAN min_cluster_size={topic_config.min_cluster_size}, "
    f"min_samples={topic_config.min_samples}."
)

In [ ]:
topic_result = run_topic_modeling_analysis(
    translated_messages,
    embedding_lookup,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    config=topic_config,
)
globals().update(topic_result.to_namespace())

print(f"Section 7 completed in {topic_result.analysis_seconds:.2f}s")
if not topic_config.label_overrides:
    print("TOPIC_LABEL_OVERRIDES is empty; labels below are keyword-based placeholders.")

display(topic_prep_summary_df)
display(topic_summary_df)
display(topic_keyword_df.loc[topic_keyword_df["rank"] <= 5])
display(topic_example_messages_df)

In [ ]:
display(topic_scatter_fig)
display(topic_prevalence_fig)
display(topic_time_fig)
display(topic_prevalence_df)

**How to read the topic figures**

- `topic_scatter_fig`: each point is a message; position is the UMAP projection, color is the cluster, and marker size scales with message length. Tight clouds suggest a coherent topic. The `Noise / Mixed` cluster contains messages that did not fit cleanly into a dense cluster.
- `topic_prevalence_fig`: bars rank topics by message count. This is a volume view, so it tells you which themes occupied the most space in the dataset.
- `topic_time_fig`: stacked daily shares show how the channel reallocated attention over time. A band grows when that topic took a larger share of that day's output, even if total daily volume also changed.


---
## Section 8 - Word Frequency & TF-IDF Shifts

This section implements `docs/03_word_frequency_tfidf_shifts.md` directly inside the notebook.

**Methodology.** The notebook cleans translated text, removes generic and channel-specific boilerplate, splits the collection window into equal temporal bins, and concatenates each bin into a pseudo-document. TF-IDF is then computed across those period documents so a term becomes important when it is unusually characteristic of one period rather than common everywhere. Rank-delta tables highlight the strongest risers and fallers between the opening and closing periods.

**What this section measures.** This is a distinctiveness analysis, not a raw frequency count. A term can be common overall and still score low if it appears in every period, while a term can score highly when it suddenly becomes period-specific.

Run Sections 1-4 first. Sections 5-7 are not required for this section.


In [ ]:
from telegram_scraper.analysis.lexical import LexicalShiftConfig, run_tfidf_shift_analysis

tfidf_config = LexicalShiftConfig()
print(
    f"Section 8 configured with {tfidf_config.periods} equal time bins and up to "
    f"{tfidf_config.max_features} TF-IDF features."
)

In [ ]:
tfidf_result = run_tfidf_shift_analysis(
    translated_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    config=tfidf_config,
)
globals().update(tfidf_result.to_namespace())

print(f"Section 8 completed in {tfidf_result.analysis_seconds:.2f}s")
display(tfidf_summary_df)
display(tfidf_period_docs_df[["period_label", "message_count", "token_count", "first_seen", "last_seen"]])
display(tfidf_movers_df)
display(tfidf_messages_df[["timestamp", "source_language", "used_translation", "token_count", "clean_text"]].head(10))

_close_matplotlib_figures(tfidf_bump_fig, tfidf_terms_fig, tfidf_wordcloud_fig)


In [ ]:
display(tfidf_bump_fig)
display(tfidf_terms_fig)

In [ ]:
display(tfidf_wordcloud_fig)

**How to read the lexical figures**

- `tfidf_bump_fig`: each line is a term across the time bins. The y-axis is TF-IDF rank, so moving upward means a term became more distinctive. Green highlights are the strongest risers; red highlights are the strongest fallers.
- `tfidf_terms_fig`: each panel is a period snapshot. Blue bars are recurring distinctive terms, green bars are new entrants relative to the prior period, and red hatched bars mark terms that were important in the previous period but dropped out of the current top set.
- `tfidf_wordcloud_fig`: use the word clouds as quick visual summaries only. Word size is based on TF-IDF, not raw frequency, so the clouds emphasize terms that are unusually characteristic of a period.


---
## Section 9 - Named Entity Network Graphs

This section implements `docs/04_named_entity_networks.md` directly inside the notebook.

**Methodology.** spaCy extracts `PERSON`, `ORG`, `GPE`, and `NORP` entities from `df_text`, then the notebook normalizes common aliases (for example `United States` → `US` and `Donald Trump` → `Trump`) and drops self-referential channel names. Every message contributes entity counts, and each unique pair of entities mentioned together becomes a co-occurrence edge. Low-frequency nodes and edges are filtered out before the graph is summarized with weighted degrees and community detection.

**What this section measures.** This is a co-mention network, not a social network. An edge means two actors appeared in the same message, which indicates editorial association or framing proximity rather than a real-world alliance.

Run Sections 4 and 6 first because this section consumes `df_text` from the sentiment/emotion stage.


In [ ]:
from telegram_scraper.analysis.entities import NamedEntityConfig, run_named_entity_analysis

entity_config = NamedEntityConfig()
print(
    f"Section 9 configured with spaCy model={entity_config.spacy_model}, "
    f"min_entity_messages={entity_config.min_message_count}, "
    f"and min_edge_weight={entity_config.min_edge_weight}."
)

In [ ]:
entity_result = run_named_entity_analysis(
    df_text,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    config=entity_config,
)
globals().update(entity_result.to_namespace())

print(f"Section 9 completed in {entity_result.analysis_seconds:.2f}s")
display(entity_extraction_summary_df)
display(entity_network_summary_df)
display(entity_summary_df.head(20))
display(entity_pair_df.loc[entity_pair_df["weight"] >= entity_config.min_edge_weight].head(20))
display(entity_community_summary_df)
display(
    entity_messages_df.sort_values(["entity_count", "timestamp"], ascending=[False, True])[
        ["timestamp", "entity_count", "entity_names_preview", "text_preview"]
    ].head(10)
)

_close_matplotlib_figures(entity_ego_fig)


In [ ]:
display(entity_top_entities_fig)
display(entity_network_fig)
if entity_ego_fig is not None:
    display(entity_ego_fig)
else:
    print("No ego networks available because the filtered graph is empty.")

**How to read the entity figures**

- `entity_top_entities_fig`: bars rank entities by the number of messages that mention them. Use it to establish the dominant cast before reading the network.
- `entity_network_fig`: node size reflects message count, node color reflects entity type, and edge thickness reflects how often two entities are co-mentioned. Dense hubs show actors that anchor many narratives; bridges show recurring associations between different actor groups.
- `entity_ego_fig`: each panel isolates one focal actor and its strongest neighbors. This is the fastest way to inspect how a single country, organization, or person is framed contextually.


---
## Section 10 - Messaging Cadence & Volume Heatmaps

This section implements `docs/05_messaging_cadence_heatmaps.md` directly inside the notebook.

**Methodology.** Using only timestamps and the `has_media` flag, the notebook resamples the selected channel into hourly and daily bins, builds a date × hour volume matrix, computes an average weekday/hour posting rhythm, and tracks the hourly share of media-bearing posts. The highest-volume hours are surfaced as candidate spikes, and optional annotations let you replace placeholders with reviewed event labels.

**Why this section is lightweight.** No NLP is required here, so it is often the fastest way to orient yourself in a new channel before running heavier semantic sections. If Section 4 has already run, translated text previews are reused for the spike tables.


In [ ]:
from telegram_scraper.analysis.cadence import MessagingCadenceConfig, run_messaging_cadence_analysis

CADENCE_EVENT_ANNOTATIONS = [
    # Replace these placeholders after reviewing cadence_top_spikes_df / cadence_spike_messages_df.
    # {"timestamp": "2026-04-10 12:00:00+00:00", "label": "Breaking-news burst around nuclear talks"},
    # {"timestamp": "2026-04-11 18:00:00+00:00", "label": "Military escalation coverage spikes"},
]

cadence_config = MessagingCadenceConfig()
print(
    f"Section 10 configured with {cadence_config.hour_freq} bins, "
    f"top {cadence_config.top_spike_candidates} spike candidates, "
    f"and {cadence_config.annotated_spikes} annotated spike cells."
)

In [ ]:
cadence_source_messages = translated_messages if "translated_messages" in globals() else raw_messages

cadence_result = run_messaging_cadence_analysis(
    cadence_source_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    event_annotations=CADENCE_EVENT_ANNOTATIONS,
    config=cadence_config,
)
globals().update(cadence_result.to_namespace())

print(f"Section 10 completed in {cadence_result.analysis_seconds:.2f}s")
if not CADENCE_EVENT_ANNOTATIONS:
    print(
        "CADENCE_EVENT_ANNOTATIONS is empty; using spike-preview placeholders. "
        "Replace them with reviewed labels for the final calendar heatmap."
    )

display(cadence_summary_df)
display(cadence_daily_summary_df)
display(cadence_top_spikes_df)
display(cadence_weekday_observation_df)
display(cadence_spike_messages_df.head(20))

_close_matplotlib_figures(
    cadence_calendar_heatmap_fig,
    cadence_structural_rhythm_fig,
    cadence_volume_media_fig,
)


In [ ]:
display(cadence_calendar_heatmap_fig)
display(cadence_structural_rhythm_fig)
display(cadence_volume_media_fig)
display(cadence_media_hourly_df.head(24))

**How to read the cadence figures**

- `cadence_calendar_heatmap_fig`: rows are dates, columns are hours, and cell values are raw message counts. This is the event view—look for isolated dark cells and numbered annotations marking spike hours.
- `cadence_structural_rhythm_fig`: each cell is the *average* number of messages for that weekday/hour combination. Use it to separate recurring editorial rhythm from one-off breaking-news spikes.
- `cadence_volume_media_fig`: orange bars show total hourly volume, the blue line shows the percentage of posts with media, and the dashed blue line is the overall media baseline. Media-share spikes above the baseline suggest visually heavy coverage, not just more posting.


---
## Section 11 - Framing & Rhetoric Analysis

This section implements `docs/06_framing_rhetoric_analysis.md` directly inside the notebook.

**Methodology.** Cleaned text-bearing messages are passed through a zero-shot MNLI classifier with a fixed rhetoric taxonomy: fear/threat, us-vs-them, call to action, victimhood, authority appeal, factual/neutral, conspiracy/suspicion, and triumphalism/strength. Each message receives a probability distribution across frames; the highest-scoring frame is kept unless confidence falls below the ambiguity threshold. Those scores are then aggregated over 12-hour windows, compared across the first vs. second half of the corpus, and paired with high-confidence examples for manual validation.

**Why validation matters.** Zero-shot framing is useful for structure, not ground truth. Treat the plots as a map of likely rhetorical emphasis and always sanity-check them with `rhetoric_example_messages_df` and `rhetoric_validation_sample_df`.

Run Sections 1-4 first. Section 6 is optional; if `sentiment_emotion_df` is available, this section also builds the rhetoric × sentiment heatmap. The first run downloads the zero-shot model into the local cache, and CPU inference can take a while on larger message sets.


In [ ]:
from telegram_scraper.analysis.framing import RhetoricFramingConfig, run_rhetoric_framing_analysis

rhetoric_config = RhetoricFramingConfig()
print(
    f"Section 11 configured with {rhetoric_config.window_freq} windows, "
    f"{rhetoric_config.model_batch_size}-message classifier batches, "
    f"and ambiguous threshold {rhetoric_config.ambiguous_threshold:.2f}."
)

In [ ]:
rhetoric_sentiment_source_df = sentiment_emotion_df if "sentiment_emotion_df" in globals() else None

rhetoric_result = run_rhetoric_framing_analysis(
    translated_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    sentiment_emotion_df=rhetoric_sentiment_source_df,
    config=rhetoric_config,
)
globals().update(rhetoric_result.to_namespace())

print(f"Section 11 completed in {rhetoric_result.analysis_seconds:.2f}s")
if rhetoric_sentiment_source_df is None:
    print("sentiment_emotion_df is unavailable; skipping the rhetoric × sentiment heatmap.")

_close_matplotlib_figures(rhetoric_examples_fig, rhetoric_sentiment_heatmap_fig)


In [ ]:
display(rhetoric_taxonomy_df)
display(rhetoric_summary_df)
display(rhetoric_label_counts_df)
display(rhetoric_half_summary_df)
display(rhetoric_example_messages_df)
display(rhetoric_validation_sample_df.head(20))
display(
    rhetoric_messages_df[
        [
            "timestamp",
            "dominant_frame",
            "confidence",
            "second_frame",
            "second_confidence",
            "text",
        ]
    ].head(10)
)

In [ ]:
display(rhetoric_over_time_fig)
display(rhetoric_transition_fig)
display(rhetoric_examples_fig)
if rhetoric_sentiment_heatmap_fig is not None:
    display(rhetoric_sentiment_heatmap_fig)
    display(rhetoric_sentiment_crosstab_df)
else:
    print("Rhetoric × sentiment heatmap unavailable because sentiment_emotion_df was not provided.")

**How to read the rhetoric figures**

- `rhetoric_over_time_fig`: stacked areas show each frame's share of average frame probability within a 12-hour window, so the chart reflects the mix of rhetorical emphasis over time rather than only hard label counts.
- `rhetoric_transition_fig`: the Sankey diagram is a half-to-half redistribution view. It preserves the first-half and second-half frame totals and shows how the overall frame mix changed; it is **not** a message-level trajectory model.
- `rhetoric_examples_fig`: each panel shows the highest-confidence example messages for a frame. Use this as the qualitative grounding step before writing claims about the chart.
- `rhetoric_sentiment_heatmap_fig` (when available): counts of messages by dominant frame and dominant sentiment, useful for spotting which frames are routinely negative, neutral, or occasionally positive.

For manual QA, inspect `rhetoric_validation_sample_df` and `rhetoric_example_messages_df`. If a category is systematically misassigned, adjust the candidate label wording in `src/telegram_scraper/analysis/framing.py` and rerun the section.


---
## Section 12 - Reply Threading & Engagement Proxy Analysis

This section implements `docs/07_reply_threading_analysis.md` directly inside the notebook.

**Methodology.** The notebook builds a directed reply graph from `reply_to_message_id`, adds every in-window message as a node, and inserts placeholder parent nodes when a reply points to a message outside the collection window. It then measures replies received, thread size, thread depth, time-to-first-reply, hourly reply rates, and simple correlates such as text length, media presence, sentiment, and optional embedding similarity for parent → reply pairs.

**What this section can and cannot say.** Replies are the closest thing this dataset has to an engagement proxy, but they reflect editorial threading as much as audience attention. Treat the statistical comparisons as exploratory signals about which messages attract follow-up structure, not as causal drivers.

Run Section 3 first. Section 4 is optional for translated previews; Sections 5-6 are optional if you want embedding-based reply similarity and sentiment-colored scatter plots.


In [ ]:
from telegram_scraper.analysis.reply_threading import ReplyThreadingConfig, run_reply_threading_analysis

reply_config = ReplyThreadingConfig()
print(
    f"Section 12 configured with top {reply_config.top_replied_messages} replied-to messages, "
    f"top {reply_config.top_threads} thread diagrams, "
    f"and {reply_config.first_reply_hist_bins} timing bins."
)

In [ ]:
reply_source_messages = translated_messages if "translated_messages" in globals() else raw_messages
reply_sentiment_source_df = sentiment_emotion_df if "sentiment_emotion_df" in globals() else None
reply_embedding_lookup = embedding_lookup if "embedding_lookup" in globals() else None

reply_result = run_reply_threading_analysis(
    reply_source_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    sentiment_emotion_df=reply_sentiment_source_df,
    embedding_lookup=reply_embedding_lookup,
    config=reply_config,
)
globals().update(reply_result.to_namespace())

print(f"Section 12 completed in {reply_result.analysis_seconds:.2f}s")
if reply_sentiment_source_df is None:
    print("sentiment_emotion_df is unavailable; coloring the scatter plot by has_media instead of sentiment.")
if reply_embedding_lookup is None:
    print("embedding_lookup is unavailable; reply-content review falls back to lexical overlap instead of embedding cosine similarity.")

_close_matplotlib_figures(
    reply_distribution_fig,
    reply_threads_fig,
    reply_scatter_fig,
    reply_timing_fig,
)


In [ ]:
display(reply_summary_df)
display(reply_feature_tests_df)
display(reply_distribution_df)
display(
    reply_thread_summary_df[
        [
            "thread_label",
            "thread_size",
            "thread_depth",
            "component_total_nodes",
            "external_node_count",
            "reply_messages",
            "messages_receiving_replies",
            "root_message_id",
            "root_in_dataset",
            "root_preview",
            "top_replied_message_id",
            "top_replied_count",
        ]
    ].head(10)
)
display(
    reply_top_replied_df[
        [
            "thread_label",
            "message_id",
            "timestamp",
            "reply_count",
            "message_depth",
            "thread_size",
            "thread_depth",
            "has_media",
            "text_preview",
        ]
    ]
)
display(reply_hourly_reply_rate_df)
display(reply_media_contingency_df)
display(reply_first_reply_timing_df.head(20))
display(reply_content_review_df.head(30))

In [ ]:
display(reply_distribution_fig)
display(reply_threads_fig)
display(reply_scatter_fig)
display(reply_timing_fig)

**How to read the reply figures**

- `reply_distribution_fig`: start here for the base rate. It shows how rare replied-to messages are and how long the tail is beyond `0`, `1`, `2`, and `3+` replies.
- `reply_threads_fig`: each panel is a tree of one large reply component. Read top-to-bottom as parent → reply flow; node color marks media presence, and edge labels show the time gap between connected messages.
- `reply_scatter_fig`: compares text length to replies received. When sentiment is available the color scale encodes sentiment; otherwise color separates media vs non-media. Treat visible patterns as exploratory rather than causal.
- `reply_timing_fig`: histogram of minutes until the first reply. Very short gaps usually indicate rolling updates, while longer gaps suggest delayed supplements, clarifications, or later follow-ups.

When writing findings, prefer references to the live tables above instead of hard-coding one run's numbers into the notebook text. That keeps Section 12 reusable when the channel, time window, or reviewed annotations change.


`reply_content_review_df["relationship_hint"]` is heuristic only. Use it to prioritize manual reading of parent/reply pairs, not as a final label set.

If `messages_replying_to_missing_parent` is non-zero in `reply_summary_df`, some threads begin outside the collection window, so reported thread depth is **window-limited** rather than full-channel history.


---
## Section 13 - Bigram / Trigram Phrase Networks

This section implements `docs/08_bigram_trigram_networks.md` directly inside the notebook.

**Methodology.** The notebook tokenizes cleaned translated text, removes stopwords and boilerplate, and scores recurring bigrams/trigrams with Pointwise Mutual Information (PMI), which favors phrases that occur together more tightly than chance would predict. Significant bigrams become directed edges in a phrase network, and a first-half / second-half split highlights which talking points were retained, dropped, or newly introduced.

**What this section measures.** High PMI does not always mean high importance; it means strong association. Always read phrase scores together with `frequency`, `message_count`, and example text before treating a phrase as editorially central.

Run Sections 1-4 first. Section 9 is optional; if `entity_summary_df` is available, the ranked bigram bar chart highlights phrases that contain tokens seen in named entities.


In [ ]:
from telegram_scraper.analysis.phrases import PhraseNetworkConfig, run_phrase_network_analysis

phrase_config = PhraseNetworkConfig()
print(
    f"Section 13 configured with bigram freq≥{phrase_config.min_bigram_freq}, "
    f"trigram freq≥{phrase_config.min_trigram_freq}, "
    f"bigram PMI≥{phrase_config.min_bigram_pmi:.1f}, "
    f"and network edge limit {phrase_config.network_edge_limit}."
)

In [ ]:
import re

phrase_entity_terms = None
if "entity_summary_df" in globals() and not entity_summary_df.empty and "entity" in entity_summary_df.columns:
    phrase_entity_terms = {
        token.lower()
        for entity in entity_summary_df["entity"].dropna().astype(str)
        for token in re.findall(r"[A-Za-z]+", entity)
        if token
    }

phrase_result = run_phrase_network_analysis(
    translated_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    entity_terms=phrase_entity_terms,
    config=phrase_config,
)
globals().update(phrase_result.to_namespace())

print(f"Section 13 completed in {phrase_result.analysis_seconds:.2f}s")
if phrase_entity_terms is None:
    print(
        "entity_summary_df is unavailable; the top-bigram bar chart uses a single non-entity style. "
        "Run Section 9 first if you want entity-aware highlighting."
    )

display(phrase_network_summary_df)
display(phrase_bigram_df.head(30))
display(phrase_trigram_df.head(20))
display(phrase_temporal_change_df.head(30))
display(
    phrase_messages_df[
        [
            "timestamp",
            "source_language",
            "used_translation",
            "token_count",
            "clean_text",
        ]
    ].head(10)
)

_close_matplotlib_figures(phrase_temporal_fig)


In [ ]:
display(phrase_network_fig)
display(phrase_bigram_bar_fig)
display(phrase_temporal_fig)
display(phrase_network_nodes_df.head(20))
display(phrase_network_edges_df.head(20))
display(phrase_temporal_bigram_df.head(20))

**How to read the phrase figures**

- `phrase_network_fig`: nodes are words, node size scales with total bigram frequency, arrow thickness scales with phrase frequency, and node color reflects detected network community. This reveals phrase neighborhoods rather than isolated keywords.
- `phrase_bigram_bar_fig`: bars rank bigrams by PMI. High PMI means the words co-occur more tightly than chance would suggest, so read it together with `frequency` and `message_count` before treating a phrase as central.
- `phrase_temporal_fig`: the left panel highlights phrases that disappear in the second half, and the right panel highlights phrases that newly appear there. Retained phrases stay gray so the visual emphasis stays on change.

Use `phrase_bigram_df` for the strongest phrase candidates, `phrase_network_edges_df` / `phrase_network_nodes_df` for the network backbone, and `phrase_temporal_change_df` for the first-half vs. second-half comparison.

If the network is too sparse or too noisy for your channel/time window, adjust `PhraseNetworkConfig` thresholds and rerun the section.


---
## Section 14 - Media vs. Text-Only Comparison

This section implements `docs/09_media_vs_text_comparison.md` directly inside the notebook.

**Methodology.** Messages are split into `Media` and `Text-only` segments from the raw `has_media` flag. The notebook then merges optional outputs from Sections 6, 7, 9, and 11 so the comparison can reuse sentiment, topic, entity, and rhetoric labels when they exist. It computes segment-level summaries, two-sample tests for continuous and categorical differences, segment-specific TF-IDF term lifts, and posting-hour distributions.

**How to interpret it.** This section asks whether media-bearing posts behave like a different editorial format rather than a random subset of messages. Differences in timing, topic mix, frame mix, or text length suggest that visual content is part of the channel's messaging strategy, not just decoration.

Run Section 3 first. Sections 6, 7, 9, and 11 are optional but recommended; if their outputs are unavailable, the dependent tables and panels fall back to placeholders.


In [ ]:
from telegram_scraper.analysis.media_vs_text import (
    MediaTextComparisonConfig,
    run_media_text_comparison_analysis,
)

media_text_config = MediaTextComparisonConfig()
print(
    f"Section 14 configured with top {media_text_config.top_topics} topics, "
    f"top {media_text_config.top_frames} frames, "
    f"top {media_text_config.top_terms} TF-IDF terms per segment, "
    f"and top {media_text_config.top_entities} entities when Section 9 data is available."
)

In [ ]:
media_text_source_messages = translated_messages if "translated_messages" in globals() else raw_messages
media_text_sentiment_source_df = sentiment_emotion_df if "sentiment_emotion_df" in globals() else None
media_text_topic_source_df = topic_messages_df if "topic_messages_df" in globals() else None
media_text_rhetoric_source_df = rhetoric_messages_df if "rhetoric_messages_df" in globals() else None
media_text_entity_source_df = entity_mentions_df if "entity_mentions_df" in globals() else None

media_text_result = run_media_text_comparison_analysis(
    media_text_source_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    sentiment_emotion_df=media_text_sentiment_source_df,
    topic_messages_df=media_text_topic_source_df,
    rhetoric_messages_df=media_text_rhetoric_source_df,
    entity_mentions_df=media_text_entity_source_df,
    config=media_text_config,
)
globals().update(media_text_result.to_namespace())

print(f"Section 14 completed in {media_text_result.analysis_seconds:.2f}s")
if media_text_sentiment_source_df is None:
    print("sentiment_emotion_df is unavailable; the comparison falls back to text length, timing, and TF-IDF term differences.")
if media_text_topic_source_df is None:
    print("topic_messages_df is unavailable; the topic table and grouped topic panel use placeholders. Run Section 7 for topic-level comparisons.")
if media_text_rhetoric_source_df is None:
    print("rhetoric_messages_df is unavailable; the rhetoric table and frame panel use placeholders. Run Section 11 for framing comparisons.")
if media_text_entity_source_df is None:
    print("entity_mentions_df is unavailable; entity comparison is skipped. Run Section 9 if you want top entity splits by message type.")

_close_matplotlib_figures(
    media_text_dashboard_fig,
    media_text_violin_fig,
    media_text_hour_density_fig,
    media_text_topic_fig,
    media_text_frame_fig,
)


In [ ]:
display(media_text_summary_df)
display(media_text_segment_summary_df)
display(media_text_stat_tests_df)
display(media_text_tfidf_terms_df)
if not media_text_topic_distribution_df.empty:
    display(media_text_topic_distribution_df)
else:
    print("No topic distribution table available because topic_messages_df was not provided.")
if not media_text_frame_distribution_df.empty:
    display(media_text_frame_distribution_df)
else:
    print("No frame distribution table available because rhetoric_messages_df was not provided.")
if not media_text_entity_distribution_df.empty:
    display(media_text_entity_distribution_df)
else:
    print("No entity distribution table available because entity_mentions_df was not provided.")
display(
    media_text_messages_df[
        [
            "timestamp",
            "has_media",
            "segment_label",
            "has_text",
            "text_length",
            "sentiment_score",
            "dominant_topic",
            "dominant_frame",
            "text",
        ]
    ].head(12)
)

In [ ]:
display(media_text_dashboard_fig)
display(media_text_violin_fig)
display(media_text_hour_density_fig)
display(media_text_topic_fig)
display(media_text_frame_fig)

**How to read the comparison figures**

- `media_text_dashboard_fig`: a compact overview that combines distribution, timing, topic, and frame panels. Use it first for pattern finding, then confirm claims with the dedicated figures and tables that follow.
- `media_text_violin_fig`: compares the full distributions of sentiment (if available) and text length, not just their means. Wider regions mark value ranges where each segment is most concentrated.
- `media_text_hour_density_fig`: compares posting-hour density by message type. Different peaks suggest that media posts and text-only posts serve different editorial roles or publication rhythms.
- `media_text_topic_fig`: grouped bars compare topic shares within each segment, helping you see which themes are disproportionately illustrated with media.
- `media_text_frame_fig`: stacked bars compare rhetorical frame counts by segment, showing whether certain persuasive strategies are more often paired with visuals.

If one of the optional upstream sections has not been run, the corresponding panel will show a placeholder; that is a notebook state issue, not a substantive result.


---
## Optional Cleanup

Run the next cell when you are done with the Telegram client.

In [ ]:
await telegram_client.disconnect()
print("Telegram client disconnected.")